# Ejercicios 4 y 5 · Model Card + Datasheet — Proyecto: guardrail de entrada sobre Aegis (NVIDIA)

**Qué hace este cuaderno:** reproduce los Ejercicios 4, 5 y 6 del cuaderno de la Dra. Gaur (Session 7) sobre **otro dataset**: *Aegis AI Content Safety Dataset 1.0* (`nvidia/Aegis-AI-Content-Safety-Dataset-1.0`), descrito en Ghosh et al. (2024), *AEGIS: Online Adaptive AI Content Safety Moderation with Ensemble of LLM Experts*, arXiv:2404.05993. Es uno de los tres datasets usados en Moraes et al. (2026), *Safeguarding Big Data Question-Answering Systems with a Two-layer Input Guardrails* (repositorio `big-qa-architecture`).

**Modelo documentado:** un clasificador ligero *safe / unsafe* (TF-IDF + regresión logística), equivalente a la "primera capa rápida" del guardrail de dos capas del paper. Las etiquetas binarias se derivan de los votos de los anotadores con **la misma regla que usa el repositorio** (`src/dataset.py`, clase `AegisDataset`).

**Cómo usarlo (Colab):**
1. Aegis 1.0 es un dataset **gated**: entra a https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-1.0 con tu cuenta y acepta los términos.
2. Crea un token de lectura en https://huggingface.co/settings/tokens y guárdalo en Colab como secreto llamado `HF_TOKEN` (icono 🔑 de la barra izquierda, con acceso habilitado para este cuaderno).
3. Ejecuta las celdas en orden. No hace falta GPU: el entrenamiento completo tarda menos de un minuto.

Todos los números del Model Card (accuracy, macro-F1, F1 de *unsafe*, AUPRC, recall por clase y por subgrupo) se calculan en la corrida. Los datos del Datasheet vienen de la tarjeta del dataset en Hugging Face y de Ghosh et al. (2024).

> ⚠️ El dataset contiene texto ofensivo y dañino (es material de *red-teaming*). Este cuaderno **no imprime ejemplos**, solo conteos y métricas.

In [1]:
# --- Instalación de dependencias e importaciones ---
%pip -q install -U datasets
import os, platform
import numpy as np, pandas as pd, sklearn
from collections import Counter
from datetime import date
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             average_precision_score)

# --- Constantes del experimento ---
SEEDS = range(5)          # mismas 5 semillas que en la clase
TEST_SIZE = 0.20          # partición 80/20 del split 'train' (además se evalúa en el split 'test' oficial)
MIN_N = 20                # tamaño mínimo de subgrupo para reportar recall desagregado
DATASET_ID = "nvidia/Aegis-AI-Content-Safety-Dataset-1.0"
CLASSES = ["unsafe", "safe"]   # mismo orden que CLASSIFICATION_LABELS en src/model.py del repo
# Los valores internos de las etiquetas siguen siendo 'safe'/'unsafe' (así vienen en el dataset);
# este diccionario solo sirve para mostrarlos en español en los textos y resultados.
ES = {"unsafe": "inseguro", "safe": "seguro"}

In [2]:
# --- Autenticación en Hugging Face (dataset gated) ---
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN cargado desde los secretos de Colab.")
except Exception as e:
    # Fuera de Colab, o sin el secreto configurado: login interactivo.
    # Si usas «Ejecutar todo», espera a completar el login antes de la siguiente celda.
    print("No se pudo leer el secreto HF_TOKEN (", type(e).__name__, "): inicia sesión manualmente.")
    from huggingface_hub import notebook_login
    notebook_login()

✅ HF_TOKEN cargado desde los secretos de Colab.


In [3]:
# --- Carga del dataset (Aegis 1.0) ---
raw = load_dataset(DATASET_ID)
train_df = raw["train"].to_pandas()
test_df = raw["test"].to_pandas()
LABEL_COLS = sorted(c for c in train_df.columns if c.startswith("labels_"))
print(raw)
print("Columnas de anotación:", LABEL_COLS)


def annotations(row):
    """Devuelve las anotaciones no vacías de una fila (una por anotador)."""
    out = []
    for c in LABEL_COLS:
        v = row[c]
        if v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip() == "":
            continue
        out.append(str(v).strip())
    return out


def repo_label(row):
    """Etiqueta binaria según la regla del repo (src/dataset.py, AegisDataset):
    'safe' si >= 2 anotaciones dicen "Safe"; en caso contrario, 'unsafe'."""
    safe = sum(1 for a in annotations(row) if a.lower() == "safe")
    return "safe" if safe >= 2 else "unsafe"


def majority_label(row):
    """Etiqueta alternativa: mayoría estricta entre las anotaciones presentes (empate -> 'unsafe')."""
    ann = annotations(row)
    safe = sum(1 for a in ann if a.lower() == "safe")
    return "safe" if ann and safe > len(ann) / 2 else "unsafe"


def categories(row):
    """Categorías de riesgo asignadas por al menos un anotador.
    Una anotación puede listar varias categorías separadas por coma."""
    cats = set()
    for a in annotations(row):
        for part in a.split(","):
            p = part.strip()
            if p and p.lower() != "safe":
                cats.add(p)
    return sorted(cats)


for df in (train_df, test_df):
    df["text"] = df["text"].fillna("")
    df["n_ann"] = df.apply(lambda r: len(annotations(r)), axis=1)
    df["label"] = df.apply(repo_label, axis=1)
    df["label_majority"] = df.apply(majority_label, axis=1)
    df["categories"] = df.apply(categories, axis=1)

# --- Conteos y descripción de los datos (se reutilizan después en el Model Card y el Datasheet) ---
N_TR, N_TE = len(train_df), len(test_df)
counts = train_df["label"].value_counts()
class_counts_txt = ", ".join(f"{ES.get(k, k)}: {v} ({v/N_TR:.1%})" for k, v in counts.items())
test_counts_txt = ", ".join(f"{ES.get(k, k)}: {v} ({v/N_TE:.1%})" for k, v in test_df["label"].value_counts().items())
ann_dist = {int(k): int(v) for k, v in sorted(Counter(train_df["n_ann"]).items())}
ann_dist_txt = ", ".join(f"{k} anotación(es): {v}" for k, v in ann_dist.items())
has_text_type = "text_type" in train_df.columns
text_type_txt = (", ".join(f"{k}: {v}" for k, v in train_df["text_type"].value_counts().items())
                 if has_text_type else "no se encontró la columna 'text_type'")
cat_counts = Counter(c for cs in train_df["categories"] for c in cs)
cat_counts_txt = ", ".join(f"{c}: {n}" for c, n in cat_counts.most_common())
rule_disagree = int((train_df["label"] != train_df["label_majority"]).sum())
single_safe = int(((train_df["n_ann"] == 1) & (train_df["label_majority"] == "safe")).sum())
no_cat_unsafe = int(((train_df["label"] == "unsafe") & (train_df["categories"].str.len() == 0)).sum())
empty_text = int((train_df["text"].str.strip() == "").sum())

print("Train / test:", N_TR, N_TE)
print("Distribución (train, regla del repo):", class_counts_txt)
print("Distribución (test, regla del repo):", test_counts_txt)
print("Anotaciones por texto (train):", ann_dist_txt)
print("Tipo de texto (train):", text_type_txt)
print("Categorías de riesgo (train, ≥1 anotador):", cat_counts_txt)
print("Textos vacíos (train):", empty_text)

# --- Auditoría de la regla de etiquetado del repositorio ---
label_audit_txt = (f"la regla del repo (>=2 votos 'Safe' -> seguro) y la mayoría estricta de las anotaciones presentes "
                   f"discrepan en {rule_disagree} de {N_TR} textos de train ({rule_disagree/N_TR:.1%}); "
                   f"{single_safe} textos tienen una única anotación 'Safe' pero la regla del repo los etiqueta 'unsafe'; "
                   f"{no_cat_unsafe} textos son 'unsafe' según la regla del repo aunque ningún anotador les asignó una categoría de riesgo")
print("Auditoría de la regla:", label_audit_txt)

# --- Contraste con fuentes externas ---
REFERENCE = {"train": 10798, "test": 1199,              # tarjeta del dataset en Hugging Face (verificar)
             "train_safe": 3550, "train_unsafe": 7248,  # 'support' en output/*-aegis.out del repo big-qa-architecture
             "label_cols_in_repo_comment": 4}           # comentario de src/dataset.py: "labels_0..labels_3"
check = lambda a, b: "COINCIDE" if a == b else "NO COINCIDE"
ref_check_txt = (f"train {N_TR} vs tarjeta {REFERENCE['train']}: {check(N_TR, REFERENCE['train'])}; "
                 f"test {N_TE} vs tarjeta {REFERENCE['test']}: {check(N_TE, REFERENCE['test'])}; "
                 f"seguro/inseguro {counts.get('safe', 0)}/{counts.get('unsafe', 0)} vs salida del repo "
                 f"{REFERENCE['train_safe']}/{REFERENCE['train_unsafe']}: "
                 f"{check((counts.get('safe', 0), counts.get('unsafe', 0)), (REFERENCE['train_safe'], REFERENCE['train_unsafe']))}; "
                 f"columnas de anotación {len(LABEL_COLS)} vs comentario del código del repo {REFERENCE['label_cols_in_repo_comment']}: "
                 f"{check(len(LABEL_COLS), REFERENCE['label_cols_in_repo_comment'])} (el propio código lee labels_0..labels_4)")
print("Contraste con fuentes:", ref_check_txt)

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

Content Moderation Extracted Annotations(…): reconstructing file:   0%|          |  0.00B / 4.91MB            

Content Moderation Extracted Annotations(…): downloading bytes:           |  0.00B            

Content Moderation Extracted Annotations(…): reconstructing file:   0%|          |  0.00B /  559kB            

Content Moderation Extracted Annotations(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10798 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1199 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['num_annotations', 'id', 'text', 'text_type', 'labels_0', 'labels_1', 'labels_2', 'labels_3', 'labels_4'],
        num_rows: 10798
    })
    test: Dataset({
        features: ['num_annotations', 'id', 'text', 'text_type', 'labels_0', 'labels_1', 'labels_2', 'labels_3', 'labels_4'],
        num_rows: 1199
    })
})
Columnas de anotación: ['labels_0', 'labels_1', 'labels_2', 'labels_3', 'labels_4']
Train / test: 10798 1199
Distribución (train, regla del repo): inseguro: 7248 (67.1%), seguro: 3550 (32.9%)
Distribución (test, regla del repo): inseguro: 792 (66.1%), seguro: 407 (33.9%)
Anotaciones por texto (train): 2 anotación(es): 12, 3 anotación(es): 9751, 4 anotación(es): 1031, 5 anotación(es): 4
Tipo de texto (train): combined: 3147, llm_response: 3131, user_message: 3099, multi_turn: 1421
Categorías de riesgo (train, ≥1 anotador): Criminal Planning/Confessions: 2482, Needs Caution: 2098, Hate/Identity Hate: 1253, Harassment: 1034,

In [4]:
# --- Experimento: 5 semillas, partición estratificada; el vectorizador TF-IDF se ajusta SOLO con train ---
def build_model(seed):
    """TF-IDF (1–2-gramas) + regresión logística, empaquetados en un solo pipeline."""
    return make_pipeline(
        TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_features=50_000, sublinear_tf=True),
        LogisticRegression(C=1.0, max_iter=2000, random_state=seed),
    )

X_all, y_all = train_df["text"], train_df["label"].values
X_off, y_off = test_df["text"], test_df["label"].values

# Subgrupos del split oficial 'test' (fijos para todas las semillas)
unsafe_off = y_off == "unsafe"
cat_masks = {c: unsafe_off & test_df["categories"].apply(lambda cs, c=c: c in cs).values
             for c in sorted({c for cs in test_df["categories"] for c in cs})}
cat_masks["(sin categoría; unsafe solo por la regla del repo)"] = unsafe_off & (test_df["categories"].str.len() == 0).values
small_groups = {c: int(m.sum()) for c, m in cat_masks.items() if m.sum() < MIN_N}   # demasiado pequeños: no se reportan
cat_masks = {c: m for c, m in cat_masks.items() if m.sum() >= MIN_N}
tt_masks = ({t: (test_df["text_type"] == t).values for t in test_df["text_type"].unique()}
            if has_text_type else {})

# Acumuladores de resultados: *_off = split 'test' oficial; sin sufijo = 20% reservado de train
res = {k: [] for k in ["acc", "f1m", "f1u", "pu", "ap", "maj",
                       "acc_off", "f1m_off", "f1u_off", "pu_off", "ap_off", "maj_off"]}
recalls, recalls_off = [], []
cat_rec, tt_f1, tt_rec = {}, {}, {}

for s in SEEDS:
    # 1) Partición 80/20 estratificada (cambia con cada semilla) y entrenamiento del modelo y del baseline
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=s)
    model = build_model(s).fit(Xtr, ytr)
    dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)   # baseline: siempre predice la clase mayoritaria
    unsafe_col = list(model.classes_).index("unsafe")
    # 2) Métricas globales en el 20% reservado y en el 'test' oficial
    for Xe, ye, sfx, rec_list in [(Xte, yte, "", recalls), (X_off, y_off, "_off", recalls_off)]:
        pred = model.predict(Xe)
        proba = model.predict_proba(Xe)[:, unsafe_col]
        res["acc" + sfx].append(accuracy_score(ye, pred))
        res["f1m" + sfx].append(f1_score(ye, pred, average="macro"))
        res["f1u" + sfx].append(f1_score(ye, pred, pos_label="unsafe"))
        res["pu" + sfx].append(precision_score(ye, pred, pos_label="unsafe"))
        res["ap" + sfx].append(average_precision_score(ye == "unsafe", proba))
        res["maj" + sfx].append(accuracy_score(ye, dummy.predict(Xe)))
        rec_list.append(recall_score(ye, pred, average=None, labels=CLASSES))
    # 3) Métricas desagregadas (solo en el 'test' oficial): por categoría de riesgo y por tipo de texto
    pred_off = model.predict(X_off)
    for c, m in cat_masks.items():
        cat_rec.setdefault(c, []).append(float((pred_off[m] == "unsafe").mean()))
    for t, m in tt_masks.items():
        tt_f1.setdefault(t, []).append(f1_score(y_off[m], pred_off[m], average="macro"))
        um = m & unsafe_off
        if um.sum() > 0:
            tt_rec.setdefault(t, []).append(float((pred_off[um] == "unsafe").mean()))

# --- Resumen: media ± desviación estándar (ddof=1) sobre las 5 semillas ---
ms = lambda a: f"{np.mean(a):.3f} ± {np.std(a, ddof=1):.3f}"
recalls, recalls_off = np.array(recalls), np.array(recalls_off)
per_class_txt = ", ".join(f"{ES[c]}: {ms(recalls[:, i])}" for i, c in enumerate(CLASSES))
per_class_off_txt = ", ".join(f"{ES[c]}: {ms(recalls_off[:, i])}" for i, c in enumerate(CLASSES))
cat_txt = "; ".join(f"{c} (n={int(cat_masks[c].sum())}): {ms(r)}"
                    for c, r in sorted(cat_rec.items(), key=lambda kv: np.mean(kv[1])))
small_txt = (f" No se reportaron las categorías con menos de {MIN_N} textos unsafe en el test: "
             + ", ".join(f"{c} (n={n})" for c, n in small_groups.items()) + ".") if small_groups else ""
tt_txt = ("; ".join(f"{t}: macro-F1 {ms(tt_f1[t])}, recall de unsafe {ms(tt_rec[t])} (n={int(tt_masks[t].sum())})"
                    for t in tt_masks if t in tt_rec)
          if has_text_type else "no se encontró la columna 'text_type': el análisis por tipo de texto NO se ejecutó.")

print("RESERVA (hold-out) 20% de train")
print("  Exactitud (accuracy):", ms(res["acc"]), "| clase mayoritaria:", ms(res["maj"]))
print("  Macro-F1            :", ms(res["f1m"]))
print("  F1 (unsafe)         :", ms(res["f1u"]), "| precisión (unsafe):", ms(res["pu"]))
print("  AUPRC (unsafe)      :", ms(res["ap"]))
print("  Recall por clase    :", per_class_txt)
print(f"SPLIT 'test' OFICIAL (n={N_TE})")
print("  Exactitud (accuracy):", ms(res["acc_off"]), "| clase mayoritaria:", ms(res["maj_off"]))
print("  Macro-F1            :", ms(res["f1m_off"]))
print("  F1 (unsafe)         :", ms(res["f1u_off"]), "| precisión (unsafe):", ms(res["pu_off"]))
print("  AUPRC (unsafe)      :", ms(res["ap_off"]))
print("  Recall por clase    :", per_class_off_txt)
print("Recall de 'unsafe' por categoría de riesgo (test oficial, de menor a mayor):\n ", cat_txt + small_txt)
print("Por tipo de texto (test oficial):", tt_txt)

RESERVA (hold-out) 20% de train
  Exactitud (accuracy): 0.772 ± 0.005 | clase mayoritaria: 0.671 ± 0.000
  Macro-F1            : 0.687 ± 0.007
  F1 (unsafe)         : 0.851 ± 0.003 | precisión (unsafe): 0.761 ± 0.003
  AUPRC (unsafe)      : 0.920 ± 0.004
  Recall por clase    : inseguro: 0.965 ± 0.003, seguro: 0.380 ± 0.009
SPLIT 'test' OFICIAL (n=1199)
  Exactitud (accuracy): 0.773 ± 0.004 | clase mayoritaria: 0.661 ± 0.000
  Macro-F1            : 0.698 ± 0.008
  F1 (unsafe)         : 0.849 ± 0.002 | precisión (unsafe): 0.759 ± 0.004
  AUPRC (unsafe)      : 0.912 ± 0.002
  Recall por clase    : inseguro: 0.962 ± 0.004, seguro: 0.406 ± 0.015
Recall de 'unsafe' por categoría de riesgo (test oficial, de menor a mayor):
  Needs Caution (n=194): 0.924 ± 0.007; Profanity (n=50): 0.936 ± 0.022; Hate/Identity Hate (n=137): 0.953 ± 0.004; PII/Privacy (n=73): 0.964 ± 0.008; Controlled/Regulated Substances (n=66): 0.970 ± 0.011; Criminal Planning/Confessions (n=298): 0.970 ± 0.004; Violence (n=1

### Referencia externa: guardrails del repositorio `big-qa-architecture`
Resultados guardados en `output/*-aegis.out` (clase *unsafe*). **No son directamente comparables**: esos modelos se usaron sin reentrenar (*zero-shot*) sobre el split `train` completo (n = 10 798), mientras que nuestro modelo se entrena con datos de Aegis y se evalúa en el split `test` oficial. Es decir, nuestro modelo tiene ventaja *in-distribution*.

In [5]:
# Resultados del repo big-qa-architecture sobre Aegis (clase 'unsafe'): precisión, recall y F1
REPO_AEGIS = pd.DataFrame([
    ("BERT (jackhhao/jailbreak-classifier)",      0.6457, 0.0581, 0.1066),
    ("DistilBERT (madhurjindal/Jailbreak-Detector)", 0.6555, 0.0459, 0.0859),
    ("ELECTRA (idanpers/JailBreakModel)",         0.7122, 0.2475, 0.3674),
    ("Samsung SGuard-JailbreakFilter-2B",         0.9296, 0.0728, 0.1351),
    ("ShieldGemma-2B",                            0.9575, 0.3201, 0.4798),
    ("Llama-Guard-3-1B",                          0.6663, 0.4432, 0.5323),
    ("Cadena BERT → Samsung",                     0.7388, 0.2947, 0.4213),
    ("Cadena BERT → ShieldGemma",                 0.8166, 0.4952, 0.6165),
    ("Cadena BERT → Llama-Guard (resultado principal del paper)", 0.6762, 0.5875, 0.6287),
], columns=["modelo", "precision_unsafe", "recall_unsafe", "f1_unsafe"])
REPO_AEGIS["conjunto_evaluacion"] = "Aegis train, zero-shot (n=10798)"

# Fila de este cuaderno: promedio de las 5 semillas sobre el 'test' oficial
ours = pd.DataFrame([{
    "modelo": "ESTE CUADERNO: TF-IDF + RegLog (promedio de 5 semillas)",
    "precision_unsafe": round(np.mean(res["pu_off"]), 4),
    "recall_unsafe": round(np.mean(recalls_off[:, CLASSES.index("unsafe")]), 4),
    "f1_unsafe": round(np.mean(res["f1u_off"]), 4),
    "conjunto_evaluacion": f"Aegis test oficial, entrenado en Aegis (n={N_TE})",
}])
comparison = pd.concat([REPO_AEGIS, ours], ignore_index=True).sort_values("f1_unsafe", ascending=False)
best_repo = REPO_AEGIS.loc[REPO_AEGIS["f1_unsafe"].idxmax()]
repo_ref_txt = (f"Mejor guardrail del repo big-qa-architecture en Aegis: {best_repo['modelo']} "
                f"(F1 unsafe {best_repo['f1_unsafe']:.4f}, zero-shot sobre Aegis train); este modelo: F1 unsafe "
                f"{ms(res['f1u_off'])} en el split test oficial. Conjuntos de evaluación distintos: no es una comparación equivalente.")
print(repo_ref_txt)
comparison

Mejor guardrail del repo big-qa-architecture en Aegis: Cadena BERT → Llama-Guard (resultado principal del paper) (F1 unsafe 0.6287, zero-shot sobre Aegis train); este modelo: F1 unsafe 0.849 ± 0.002 en el split test oficial. Conjuntos de evaluación distintos: no es una comparación equivalente.


,modelo,precision_unsafe,recall_unsafe,f1_unsafe,conjunto_evaluacion
9,ESTE CUADERNO: TF-IDF + RegLog (promedio de 5 semillas),0.7591,0.9619,0.8485,"Aegis test oficial, entrenado en Aegis (n=1199)"
8,Cadena BERT → Llama-Guard (resultado principal del paper),0.6762,0.5875,0.6287,"Aegis train, zero-shot (n=10798)"
7,Cadena BERT → ShieldGemma,0.8166,0.4952,0.6165,"Aegis train, zero-shot (n=10798)"
5,Llama-Guard-3-1B,0.6663,0.4432,0.5323,"Aegis train, zero-shot (n=10798)"
4,ShieldGemma-2B,0.9575,0.3201,0.4798,"Aegis train, zero-shot (n=10798)"
6,Cadena BERT → Samsung,0.7388,0.2947,0.4213,"Aegis train, zero-shot (n=10798)"
2,ELECTRA (idanpers/JailBreakModel),0.7122,0.2475,0.3674,"Aegis train, zero-shot (n=10798)"
3,Samsung SGuard-JailbreakFilter-2B,0.9296,0.0728,0.1351,"Aegis train, zero-shot (n=10798)"
0,BERT (jackhhao/jailbreak-classifier),0.6457,0.0581,0.1066,"Aegis train, zero-shot (n=10798)"
1,DistilBERT (madhurjindal/Jailbreak-Detector),0.6555,0.0459,0.0859,"Aegis train, zero-shot (n=10798)"


## Ejercicio 4 · Model Card (9 secciones)
Mismas claves y estructura que el cuaderno de la profesora (el scorecard cuenta exactamente 9 secciones). Los números vienen de las celdas anteriores.

In [6]:
# --- Model Card (9 secciones). Los números provienen de las celdas anteriores ---
model_card = {
    "Detalles del modelo": {
        "Nombre / versión": "AegisSafety-TFIDF-LR v0.1",
        "Responsable / contacto": "FELIX RICHAR MENDOZA PUMACAHUA <felix.mendozap@unmsm.edu.pe>",
        "Fecha": date.today().isoformat(),
        "Arquitectura": "TF-IDF (1–2-gramas de palabras, min_df=2, máximo 50,000 atributos, tf sublineal) + regresión "
                        "logística con regularización L2 (C=1.0); salida binaria seguro/inseguro con umbral de probabilidad 0.5; "
                        f"scikit-learn {sklearn.__version__}, Python {platform.python_version()}",
        "Licencia": "Código: MIT (propuesta por el autor). Datos: CC BY 4.0 según la tarjeta del dataset en Hugging Face "
                    "(verificar); el acceso está restringido por los términos de NVIDIA.",
    },
    "Uso previsto": {
        "Usos previstos principales": "Ejercicio del curso (Métodos de Investigación, UNMSM) para practicar la documentación de un "
                                      "clasificador de seguridad de contenido. Línea base de investigación para la primera capa "
                                      "rápida de un guardrail de entrada de dos capas (Moraes et al., 2026) en sistemas de "
                                      "preguntas y respuestas sobre Big Data.",
        "Fuera de alcance / mal uso": "NO es un filtro de seguridad para producción y NO debe usarse para moderar a usuarios reales "
                                      "ni para tomar decisiones sobre personas. NO es un detector de jailbreaks: Aegis etiqueta "
                                      "CONTENIDO dañino, no TÉCNICAS de jailbreak (juego de roles, ofuscación, inyección de "
                                      "prompts). Solo inglés. Un modelo de bolsa de palabras se evade trivialmente con paráfrasis, "
                                      "errores ortográficos o codificación.",
        "Usuarios previstos": "Estudiantes de doctorado de este curso; revisores de este ejercicio.",
    },
    "Factores": {
        "Grupos / condiciones relevantes": "Categoría de riesgo del texto (taxonomía de Aegis: 13 categorías críticas más 'Needs "
                                           "Caution'); tipo de texto (prompt de usuario vs respuesta del LLM"
                                           + ("" if has_text_type else " — columna no encontrada") + "); idioma (solo inglés); "
                                           "origen (prompts de red-teaming, adversarios por construcción); desacuerdo entre "
                                           "anotadores (número variable de anotaciones por texto). Los grupos identitarios "
                                           "mencionados en el texto NO están anotados, por lo que el sesgo hacia términos de "
                                           "identidad no se puede medir con este dataset (DESCONOCIDO — por investigar).",
        "Factores de evaluación": "Se reporta en total, por clase, por categoría de riesgo (recall de 'unsafe') y por tipo de texto.",
    },
    "Métricas": {
        "Medidas de desempeño": "Exactitud (accuracy), macro-F1 y F1/precisión de la clase 'unsafe' (las clases están desbalanceadas, "
                                "así que la exactitud sola engaña); AUPRC de la probabilidad de 'unsafe' (no depende del umbral); "
                                "recall por clase. Recall de 'unsafe' = proporción de textos dañinos que el guardrail detiene.",
        "Umbrales de decisión": "0.5 sobre la probabilidad predicha (sin ajuste de umbral).",
        "Enfoque de variación": f"Media ± desviación estándar (ddof=1) sobre {len(list(SEEDS))} semillas aleatorias; cada semilla "
                                "cambia la partición estratificada 80/20 de 'train' de Aegis y el random_state del modelo.",
    },
    "Datos de evaluación": {
        "Datasets": f"(a) {int(TEST_SIZE*100)}% reservado y estratificado del 'train' de Aegis, redibujado en cada semilla; "
                    f"(b) el split 'test' oficial de Aegis ({N_TE} textos; {test_counts_txt}), fijo para todas las semillas.",
        "Preprocesamiento": "Etiquetas binarias derivadas de los votos de los anotadores con la regla del repo (seguro si >= 2 votos "
                            "'Safe'). El vocabulario TF-IDF se ajusta solo con el split de entrenamiento (sin fuga de datos hacia el test).",
    },
    "Datos de entrenamiento": {
        "Datasets": f"{int((1-TEST_SIZE)*100)}% del 'train' de Aegis 1.0 ({N_TR} textos en total; clases: {class_counts_txt}).",
        "Limitaciones conocidas": f"Las etiquetas dependen de la regla de binarización: {label_audit_txt}. "
                                  f"Anotaciones por texto: {ann_dist_txt}. Los prompts y las respuestas del LLM están mezclados en "
                                  f"un mismo conjunto ({text_type_txt}). Las categorías de riesgo están muy desbalanceadas ({cat_counts_txt}).",
    },
    "Análisis cuantitativos": {
        "Resultados generales": f"Reserva (hold-out): exactitud={ms(res['acc'])}, macro-F1={ms(res['f1m'])}, F1 unsafe={ms(res['f1u'])}, "
                                f"AUPRC={ms(res['ap'])} (exactitud de la clase mayoritaria como referencia: {ms(res['maj'])}). "
                                f"Test oficial: exactitud={ms(res['acc_off'])}, macro-F1={ms(res['f1m_off'])}, "
                                f"F1 unsafe={ms(res['f1u_off'])}, AUPRC={ms(res['ap_off'])} "
                                f"(exactitud de la clase mayoritaria: {ms(res['maj_off'])}). {repo_ref_txt}",
        "Resultados desagregados": f"recall por clase (reserva): {per_class_txt}; recall por clase (test oficial): "
                                   f"{per_class_off_txt}. Recall de 'unsafe' por categoría de riesgo (test oficial): "
                                   f"{cat_txt}.{small_txt} Por tipo de texto: {tt_txt}.",
    },
    "Consideraciones éticas": {
        "Riesgos": "Los falsos negativos dejan pasar solicitudes dañinas hacia el LLM y la base de conocimiento; los falsos positivos "
                   "bloquean en exceso preguntas legítimas y pueden recaer desproporcionadamente sobre textos que mencionan grupos "
                   "identitarios o temas sensibles (algo común en clasificadores de toxicidad; no se midió aquí). Los datos de "
                   "entrenamiento contienen contenido ofensivo. Un modelo lineal transparente puede analizarse para encontrar "
                   "formas de evadirlo (doble uso).",
        "Mitigaciones": "Uso solo educativo; sin despliegue; nunca se imprimen ejemplos. Cualquier uso real requeriría una segunda "
                        "capa más robusta (p. ej., un guardrail basado en LLM), una auditoría de sesgo por términos de identidad, "
                        "pruebas adversariales y revisión humana (DESCONOCIDO — por investigar).",
    },
    "Salvedades y recomendaciones": {
        "Salvedades": "Evaluación solo dentro de la distribución (misma fuente y taxonomía que el entrenamiento). Los resultados NO se "
                      "trasladarán a tráfico real de preguntas y respuestas, a otros idiomas ni a ataques tipo jailbreak. Sin ajuste de "
                      "hiperparámetros; la comparación con los guardrails del repo usa conjuntos de evaluación distintos.",
        "Recomendaciones": "Evaluar los guardrails del repo sobre el mismo split test oficial; probar fuera de distribución con los "
                           "prompts de jailbreak reales de TrustAIRLab (in-the-wild); revisar la calibración y ajustar el umbral para "
                           "el rol de primera capa (alta precisión); reportar resultados también con la regla de mayoría estricta; "
                           "realizar una auditoría de sesgo por términos de identidad antes de afirmar utilidad alguna.",
    },
}

def render_card(card, title):
    """Convierte el diccionario (secciones -> campos) en Markdown."""
    lines = [f"# {title}", ""]
    for section, fields in card.items():
        lines.append(f"## {section}")
        for k, v in fields.items():
            lines.append(f"- **{k}:** {v}")
        lines.append("")
    return "\n".join(lines)

card_md = render_card(model_card, "Model Card — Guardrail de entrada de seguridad de contenido sobre Aegis (uso educativo)")
print(card_md)
with open("model_card.md", "w", encoding="utf-8") as f:
    f.write(card_md)
print("\n✅ Guardado model_card.md")

# Model Card — Guardrail de entrada de seguridad de contenido sobre Aegis (uso educativo)

## Detalles del modelo
- **Nombre / versión:** AegisSafety-TFIDF-LR v0.1
- **Responsable / contacto:** FELIX RICHAR MENDOZA PUMACAHUA <felix.mendozap@unmsm.edu.pe>
- **Fecha:** 2026-10-30
- **Arquitectura:** TF-IDF (1–2-gramas de palabras, min_df=2, máximo 50,000 atributos, tf sublineal) + regresión logística con regularización L2 (C=1.0); salida binaria seguro/inseguro con umbral de probabilidad 0.5; scikit-learn 1.6.1, Python 3.13.18
- **Licencia:** Código: MIT (propuesta por el autor). Datos: CC BY 4.0 según la tarjeta del dataset en Hugging Face (verificar); el acceso está restringido por los términos de NVIDIA.

## Uso previsto
- **Usos previstos principales:** Ejercicio del curso (Métodos de Investigación, UNMSM) para practicar la documentación de un clasificador de seguridad de contenido. Línea base de investigación para la primera capa rápida de un guardrail de entrada de dos capas (Morae

## Ejercicio 5 · Datasheet (7 secciones)
Lo tomado de la tarjeta del dataset en Hugging Face y de Ghosh et al. (2024) está escrito como hecho y marcado *(verificar)* cuando hay que confirmarlo en la fuente; lo que esas fuentes no dicen queda como `DESCONOCIDO — por investigar`. Los conteos se calculan en la corrida.

In [7]:
# --- Datasheet (7 secciones) ---
# Lo tomado de la tarjeta del dataset y de Ghosh et al. (2024) se escribe como hecho y se marca (verificar) si hay que confirmarlo;
# lo que esas fuentes no dicen queda como "DESCONOCIDO — por investigar". Los conteos se calculan en la corrida.
datasheet = {
    "Motivación": {
        "Propósito": "Creado para entrenar y evaluar modelos de moderación de seguridad de contenido de LLM frente a una taxonomía "
                     "amplia de riesgos, incluido el ensamble AEGIS de expertos LLM de seguridad (Ghosh et al., 2024).",
        "Creado por / financiado por": "NVIDIA (Ghosh, Varshney, Galinkin, Parisien — verificar la lista de autores en arXiv:2404.05993). "
                                       "Financiamiento: interno de NVIDIA; subvención o financiamiento externo DESCONOCIDO — por investigar.",
    },
    "Composición": {
        "Qué representan las instancias": f"Una fila por texto, que es un prompt de usuario o una respuesta de LLM ({text_type_txt}). "
                                          f"Tamaño de los splits: {N_TR} train / {N_TE} test. Cada fila tiene hasta {len(LABEL_COLS)} "
                                          f"columnas de anotación ({', '.join(LABEL_COLS)}); cada anotación es 'Safe' o una o "
                                          f"más categorías de riesgo. Anotaciones por texto: {ann_dist_txt}. "
                                          f"Contraste con fuentes en esta corrida: {ref_check_txt}.",
        "Etiquetas": "Etiquetas por anotador de la taxonomía de Aegis (13 categorías de riesgo críticas más 'Needs Caution'; "
                     "verificar la lista en la tarjeta del dataset). Frecuencia de categorías en train (asignadas por >= 1 anotador): "
                     f"{cat_counts_txt}. Etiqueta binaria usada aquí = regla del repo (seguro si >= 2 votos 'Safe'): {class_counts_txt} "
                     "(desbalanceada). Los creadores no proporcionan una etiqueta binaria oficial.",
        "Muestra vs población": "Prompts en inglés tomados de los datos de red-teaming de inocuidad HH-RLHF de Anthropic, más respuestas "
                                "generadas por un LLM (Mistral-7B-v0.1 según la tarjeta del dataset — verificar). Adversarial por "
                                "construcción; NO es representativo del tráfico real de usuarios, de otros idiomas ni de preguntas de "
                                "QA sobre documentos empresariales.",
        "Datos faltantes": f"{empty_text} textos vacíos en train. Es esperable que haya espacios de anotación vacíos porque los textos "
                           f"tienen un número variable de anotadores ({ann_dist_txt}); el motivo por el que algunos textos tienen menos "
                           "anotaciones es DESCONOCIDO — por investigar.",
    },
    "Proceso de recolección": {
        "Cómo se recolectó": "Prompts muestreados de Anthropic HH-RLHF (harmless-base, conversaciones de red-teaming); respuestas de LLM "
                             "generadas por los creadores; cada texto fue etiquetado por anotadores humanos capacitados siguiendo la "
                             "taxonomía y las guías de Aegis (Ghosh et al., 2024 — verificar el número de anotadores y el procedimiento de control de calidad).",
        "Periodo": "Publicado en 2024 junto con arXiv:2404.05993 (abril de 2024). Periodo de anotación DESCONOCIDO — por investigar.",
        "Consentimiento / ética": "Los prompts de HH-RLHF fueron escritos por trabajadores colaborativos (crowdworkers) contratados por Anthropic; "
                                  "sus términos de consentimiento son los de la recolección original (DESCONOCIDO — por investigar). "
                                  "Los anotadores estuvieron expuestos a contenido dañino; las medidas de bienestar se describen en el "
                                  "artículo (verificar). Aprobación de un comité de ética: DESCONOCIDO — por investigar.",
    },
    "Preprocesamiento / limpieza / etiquetado": {
        "Pasos": "Filtrado, deduplicación y tratamiento de PII por parte de los creadores: DESCONOCIDO — por investigar. En este proyecto: "
                 "(1) etiqueta binaria derivada con la regla de src/dataset.py de big-qa-architecture; "
                 f"(2) auditoría de la regla de etiquetado: {label_audit_txt}; (3) el comentario del código del repo menciona labels_0..labels_3 "
                 "pero el código lee labels_0..labels_4; (4) atributos TF-IDF al momento del entrenamiento (ajustados solo con train).",
        "Datos crudos disponibles": "Se publican las etiquetas por anotador (columnas labels_*), de modo que se pueden aplicar otras reglas de "
                                    "agregación. Las conversaciones completas de HH-RLHF son públicas por separado (Anthropic, licencia MIT — "
                                    "verificar). Guías completas para anotadores: DESCONOCIDO — por investigar.",
    },
    "Usos": {
        "Usos apropiados": "Entrenar y comparar clasificadores de seguridad de contenido y guardrails de LLM; investigación sobre el "
                           "desacuerdo entre anotadores y la agregación de etiquetas; enseñanza de documentación de IA responsable.",
        "No debe usarse para": "Entrenar modelos que GENEREN contenido dañino; afirmar desempeño en detección de jailbreaks sin pruebas "
                               "fuera de distribución (las etiquetas describen categorías de daño, no técnicas de ataque); perfilar o "
                               "tomar decisiones sobre personas reales; generalizar a texto que no esté en inglés.",
    },
    "Distribución": {
        "Cómo se distribuye": "Hugging Face Hub, con acceso restringido (los usuarios deben aceptar los términos de NVIDIA): "
                              "https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-1.0. "
                              "Artículo del dataset: Ghosh et al. (2024), arXiv:2404.05993.",
        "Licencia / restricciones": "CC BY 4.0 según la tarjeta del dataset (verificar): se requiere atribución; acceso restringido; la "
                                    "tarjeta advierte que los datos contienen contenido ofensivo.",
    },
    "Mantenimiento": {
        "Responsable": "NVIDIA, mediante la página del dataset en Hugging Face; existe un dataset sucesor (Aegis 2.0) (verificar). "
                       "Compromiso formal de mantenimiento de la v1.0: DESCONOCIDO — por investigar.",
        "Plan de actualización": "Ninguno declarado para la v1.0 en la página del dataset ni en el artículo (verificar).",
    },
}

sheet_md = render_card(datasheet, "Datasheet — NVIDIA Aegis AI Content Safety Dataset 1.0 (uso educativo)")
print(sheet_md)
with open("datasheet.md", "w", encoding="utf-8") as f:
    f.write(sheet_md)
print("\n✅ Guardado datasheet.md")

# Datasheet — NVIDIA Aegis AI Content Safety Dataset 1.0 (uso educativo)

## Motivación
- **Propósito:** Creado para entrenar y evaluar modelos de moderación de seguridad de contenido de LLM frente a una taxonomía amplia de riesgos, incluido el ensamble AEGIS de expertos LLM de seguridad (Ghosh et al., 2024).
- **Creado por / financiado por:** NVIDIA (Ghosh, Varshney, Galinkin, Parisien — verificar la lista de autores en arXiv:2404.05993). Financiamiento: interno de NVIDIA; subvención o financiamiento externo DESCONOCIDO — por investigar.

## Composición
- **Qué representan las instancias:** Una fila por texto, que es un prompt de usuario o una respuesta de LLM (combined: 3147, llm_response: 3131, user_message: 3099, multi_turn: 1421). Tamaño de los splits: 10798 train / 1199 test. Cada fila tiene hasta 5 columnas de anotación (labels_0, labels_1, labels_2, labels_3, labels_4); cada anotación es 'Safe' o una o más categorías de riesgo. Anotaciones por texto: 2 anotación(es): 12, 3 anot

## Ejercicio 6 · Scorecard (autoauditoría)
Misma lógica que la función de la profesora, ampliada para aceptar palabras clave en inglés y en español (los documentos de este cuaderno están redactados en español). El objetivo es **8/8**.

In [8]:
def score_documentation(model_card, datasheet):
    """Autoauditoría de 8 controles sobre el Model Card y el Datasheet.
    Misma lógica que la función de la profesora; cada control acepta palabras clave en inglés Y en español,
    porque los documentos de este cuaderno están redactados en español."""
    checks = []
    def add(label, passed, why=""):
        checks.append((label, bool(passed), why))
    def flatten(card):
        # Une en un solo texto en minúsculas todos los títulos de campo y sus contenidos
        parts = []
        for fields in card.values():
            for k, v in fields.items():
                parts.append(str(k).lower()); parts.append(str(v).lower())
        return " ".join(parts)
    has_any = lambda text, words: any(w in text for w in words)
    flat_mc = flatten(model_card); flat_ds = flatten(datasheet)
    add("El Model Card tiene las 9 secciones", len(model_card) == 9)
    add("El Datasheet tiene las 7 secciones", len(datasheet) == 7)
    add("Se declara el MAL USO / uso fuera de alcance",
        has_any(flat_mc, ["not for", "out-of-scope", "should not", "fuera de alcance", "no debe usarse"]))
    add("Desempeño reportado con variación (± / desviación / semillas)",
        has_any(flat_mc, ["±", "std", "seed", "desviación", "semilla"]))
    add("Métricas desagregadas (por clase / por grupo)",
        has_any(flat_mc, ["per-class", "per class", "disaggreg", "por clase", "desagregad"]))
    add("Se declara para qué el dataset NO debe usarse",
        has_any(flat_ds, ["should not be used", "not for", "no debe usarse", "no debe usarse para"]))
    coll = datasheet.get("Collection process") or datasheet.get("Proceso de recolección") or {}
    add("Se describe el proceso de recolección",
        len(coll) > 0 and has_any(" ".join(coll.keys()).lower(), ["how collected", "cómo se recolectó"]))
    add("Se señalan honestamente los vacíos ('desconocido')",
        has_any(flat_mc + " " + flat_ds, ["unknown", "desconocido"]),
        "Señalar lo que no sabes es mejor que dejar un vacío con falsa seguridad.")
    passed = sum(1 for _, p, _ in checks if p)
    print("TARJETA DE PUNTUACIÓN DE LA DOCUMENTACIÓN"); print("=" * 52)
    for label, p, why in checks:
        print("[" + ("✓" if p else "✗") + "] " + label)
        if why and not p: print("      pista: " + why)
    print("=" * 52); print(f"Puntaje: {passed}/{len(checks)}  ({100*passed/len(checks):.0f}%)")
    return passed, len(checks)

_ = score_documentation(model_card, datasheet)

TARJETA DE PUNTUACIÓN DE LA DOCUMENTACIÓN
[✓] El Model Card tiene las 9 secciones
[✓] El Datasheet tiene las 7 secciones
[✓] Se declara el MAL USO / uso fuera de alcance
[✓] Desempeño reportado con variación (± / desviación / semillas)
[✓] Métricas desagregadas (por clase / por grupo)
[✓] Se declara para qué el dataset NO debe usarse
[✓] Se describe el proceso de recolección
[✓] Se señalan honestamente los vacíos ('desconocido')
Puntaje: 8/8  (100%)


In [ ]:
# (Opcional) Descargar los dos archivos generados
try:
    from google.colab import files
    files.download("model_card.md")
    files.download("datasheet.md")
except ImportError:
    print("Fuera de Colab: los archivos quedaron en el directorio de trabajo.")

## Checklist final antes de entregar
- [ ] Revisé la tarjeta del dataset y Ghosh et al. (2024) y confirmé cada dato marcado *(verificar)*: licencia, autores, origen de prompts y respuestas, número de anotadores, lista de categorías, medidas de bienestar de anotadores, existencia de Aegis 2.0.
- [ ] Revisé la salida de «Contraste con fuentes» (tamaños de split, conteos safe/unsafe frente al repo, columnas de anotación) y anoté cualquier diferencia.
- [ ] Revisé la «Auditoría de la regla» y decidí si la regla del repo (≥ 2 votos "Safe") es aceptable o si conviene reportar también la mayoría estricta.
- [ ] Confirmé que el Model Card dice explícitamente que Aegis mide **contenido dañino**, no **técnicas de jailbreak**.
- [ ] Scorecard 8/8.
- [ ] Subí `model_card.md` y `datasheet.md` al GitHub (carpeta del entregable 7) y/o a la plataforma del curso.

**Referencias:** Mitchell et al. (2019), *Model Cards for Model Reporting*, arXiv:1810.03993 · Gebru et al. (2021), *Datasheets for Datasets*, CACM 64(12) · Ghosh, Varshney, Galinkin y Parisien (2024), *AEGIS: Online Adaptive AI Content Safety Moderation with Ensemble of LLM Experts*, arXiv:2404.05993 · NVIDIA, *Aegis AI Content Safety Dataset 1.0*, Hugging Face · Moraes, Onofre y Aguiar (2026), *Safeguarding Big Data Question-Answering Systems with a Two-Layer Input Guardrails*, ICEIS 2026, DOI 10.5220/0014692300004018 · Bai et al. (2022), *Training a Helpful and Harmless Assistant with RLHF*, arXiv:2204.05862.